# 14 — Triangle inequality spot-check

Uses **Euclidean** distance on `[unit directions | log1p(Inf.Hyp.Rad)]`; a metric, so violations should be **~0** (floating-point tolerance only). Swap in another distance if you want a nontrivial diagnostic.


### Triangle inequality check (Euclidean feature distance)

**Method:** Random triples of vertices; distances are Euclidean on **[unit direction | log1p(Inf.Hyp.Rad)]**. Count violations of `d(a,b)+d(b,c)≥d(a,c)` (with a tiny numerical tolerance).

**How to read the output:** For true **metrics**, violation rate should be **~0** (only float noise). A nonzero rate would indicate a bug or a non-metric distance. If you swap in cosine “distance” or other non-metrics, violations can appear—then the rate is a diagnostic of how badly the triangle inequality fails.


In [ ]:
from pathlib import Path
import numpy as np

import dmercator3d_io as dm

merged = dm.load_merged_parquet(Path("cache/merged.parquet"))
rng = np.random.default_rng(11)
U = dm.normalize_direction_nd(merged)
X = np.hstack([U, np.log1p(merged["Inf.Hyp.Rad"]).to_numpy()[:, None]])
n = X.shape[0]


def d(i, j):
    return float(np.linalg.norm(X[i] - X[j]))


viol = 0
trials = 5000
for _ in range(trials):
    a, b, c = rng.integers(0, n, size=3)
    da, db, dc = d(a, b), d(b, c), d(a, c)
    if da + db < dc - 1e-6 or da + dc < db - 1e-6 or db + dc < da - 1e-6:
        viol += 1
print("violation rate:", viol / trials)
